# 19 · Калибровка и топ-K операционная точка (прод-пилот)

На holdout ≥ 2025-07-01 (20k субъектов на задачу) для трёх TTE-моделей:
1. **Калибровка** `P(событие ≤ 24ч)` — O/E по децилям predicted `p24`,
   доля фактических событий в бинах vs средняя вероятность.
2. **Топ-K операционная точка**: отсортировать по `p24` убыв., для K показать
   precision@K, recall@K (от всех событий 24ч в holdout) — это картина
   «открыть топ-100 → ловим 3 события, из них 1 истинное».
3. Выбор K при `precision ≥ 0.7` (цель ТЗ) или компромисс «максимум событий
   при приемлемом шуме» — выдаём рекомендацию для диспетчера.

Задачи: fire (серьёзных), access, sensor. Модели обучены на новых дефолтах
(1000/0.03/10).


In [1]:
%matplotlib inline
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import data_utils as du
from tte_experiments import pr_at_horizon

DATASET = du.find_dataset_dir()
sns.set_theme(style="whitegrid")

TASKS = {"fire": "Пожарный риск", "access": "Несанкц. доступ",
         "sensor": "Отказ датчика"}
LOAD = {}
for t in TASKS:
    df = pd.read_parquet(DATASET.parent / "models" / f"tte_{t}_holdout.parquet")
    y24 = ((df["event_flag"] == 1) & (df["obs_days"] <= 1.01)).to_numpy(int)
    LOAD[t] = (df, y24)
    print(f"{t}: {len(df):,} субъектов | событий 24ч: {int(y24.sum()):,} "
          f"({y24.mean()*100:.2f}%)")


fire: 19,999 субъектов | событий 24ч: 306 (1.53%)
access: 19,999 субъектов | событий 24ч: 202 (1.01%)
sensor: 19,999 субъектов | событий 24ч: 61 (0.31%)


In [2]:
def calib_oe(df, y24, bins=10):
    q = pd.qcut(df["p24"], bins, duplicates="drop")
    g = pd.DataFrame({"bin": q, "pred": df["p24"], "y": y24})
    tab = g.groupby("bin", observed=True).agg(
        n=("pred", "size"), pred=("pred", "mean"), obs=("y", "mean")).reset_index()
    tab["O/E"] = tab["obs"] / tab["pred"].replace(0, np.nan)
    return tab

print("=== Калибровка P(24ч): фактический риск vs предсказанный (децили) ===")
for t, name in TASKS.items():
    tab = calib_oe(*LOAD[t])
    print(f"--- {name} ({t}) ---")
    print(tab.round(4).to_string(index=False))
    print()


=== Калибровка P(24ч): фактический риск vs предсказанный (децили) ===
--- Пожарный риск (fire) ---
                   bin    n   pred    obs    O/E
(-0.0009404, 0.000373] 2000 0.0002 0.0000 0.0000
  (0.000373, 0.000647] 2021 0.0005 0.0000 0.0000
     (0.000647, 0.001] 1979 0.0008 0.0000 0.0000
       (0.001, 0.0018] 2023 0.0013 0.0000 0.0000
      (0.0018, 0.0035] 2034 0.0026 0.0039 1.5074
     (0.0035, 0.00657] 1987 0.0052 0.0096 1.8550
     (0.00657, 0.0113] 1956 0.0085 0.0107 1.2574
      (0.0113, 0.0168] 2004 0.0142 0.0050 0.3513
      (0.0168, 0.0307] 2003 0.0232 0.0155 0.6664
       (0.0307, 0.902] 1992 0.0845 0.1089 1.2896

--- Несанкц. доступ (access) ---
                   bin    n   pred    obs    O/E
(-0.0009074, 0.000659] 2000 0.0005 0.0005 1.0294
  (0.000659, 0.000957] 2000 0.0008 0.0015 1.8600
    (0.000957, 0.0013] 2000 0.0011 0.0020 1.7871
     (0.0013, 0.00176] 2000 0.0015 0.0030 1.9831
    (0.00176, 0.00242] 2000 0.0021 0.0050 2.4309
    (0.00242, 0.00349] 1999 0.0029

In [3]:
def topk(df, y24, ks):
    order = np.argsort(-df["p24"].to_numpy())
    y_sorted = y24[order]
    cum = np.cumsum(y_sorted)
    total = cum[-1]
    rows = []
    for k in ks:
        k = min(k, len(y_sorted))
        hits = cum[k - 1]
        rows.append({"K": k, "precision": hits / k,
                     "recall": hits / total if total else np.nan})
    return pd.DataFrame(rows), total

KS = [10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000]
for t, name in TASKS.items():
    res, total = topk(LOAD[t][0], LOAD[t][1], KS)
    print(f"=== {name} ({t}) | всего событий 24ч в топ-пространстве: {int(total)} ===")
    print(res.round(4).to_string(index=False))
    print()


=== Пожарный риск (fire) | всего событий 24ч в топ-пространстве: 306 ===
    K  precision  recall
   10     1.0000  0.0327
   20     1.0000  0.0654
   50     1.0000  0.1634
  100     0.9700  0.3170
  200     0.5350  0.3497
  500     0.2340  0.3824
 1000     0.1450  0.4739
 2000     0.1090  0.7124
 5000     0.0498  0.8137
10000     0.0298  0.9739

=== Несанкц. доступ (access) | всего событий 24ч в топ-пространстве: 202 ===
    K  precision  recall
   10     0.0000  0.0000
   20     0.0000  0.0000
   50     0.0000  0.0000
  100     0.0100  0.0050
  200     0.0300  0.0297
  500     0.0500  0.1238
 1000     0.0450  0.2228
 2000     0.0335  0.3317
 5000     0.0272  0.6733
10000     0.0178  0.8812

=== Отказ датчика (sensor) | всего событий 24ч в топ-пространстве: 61 ===
    K  precision  recall
   10     0.1000  0.0164
   20     0.0500  0.0164
   50     0.0200  0.0164
  100     0.0100  0.0164
  200     0.0450  0.1475
  500     0.0200  0.1639
 1000     0.0150  0.2459
 2000     0.0090  0.2951

In [4]:
# Рекомендуемая операционная точка: максимальное K с precision >= 0.7 (цель ТЗ),
# и «рабочий» компромисс при precision >= 0.5.
print("=== Операционная точка по ТЗ (precision >= 0.7) ===")
for t, name in TASKS.items():
    df, y24 = LOAD[t]
    order = np.argsort(-df["p24"].to_numpy())
    ys = y24[order]
    cum = np.cumsum(ys)
    total = cum[-1]
    best = None
    for k in range(1, min(len(ys), 20000) + 1):
        p = cum[k - 1] / k
        if p >= 0.7:
            best = k
        else:
            break
    # плюс компромисс p>=0.5
    best5 = None
    for k in range(1, min(len(ys), 20000) + 1):
        if cum[k - 1] / k >= 0.5:
            best5 = k
        else:
            break
    def row(k):
        if k is None:
            return "недостижимо (база слишком мала)"
        p = cum[k-1]/k
        return (f"K={k}, precision={p:.3f}, recall={cum[k-1]/total:.3f}, "
                f"событий покрыто={int(cum[k-1])}")
    print(f"{name} ({t}):")
    print("   prec>=0.7:", row(best))
    print("   prec>=0.5:", row(best5))
print()
print("Примечание: база событий в 24ч мала (1.5%/1.0%/0.3%), поэтому K с prec>=0.7")
print("будут маленькими; для диспетчера главное — топ-K с известной ожидаемой долей.")


=== Операционная точка по ТЗ (precision >= 0.7) ===
Пожарный риск (fire):
   prec>=0.7: K=152, precision=0.704, recall=0.350, событий покрыто=107
   prec>=0.5: K=222, precision=0.500, recall=0.363, событий покрыто=111
Несанкц. доступ (access):
   prec>=0.7: недостижимо (база слишком мала)
   prec>=0.5: недостижимо (база слишком мала)
Отказ датчика (sensor):
   prec>=0.7: недостижимо (база слишком мала)
   prec>=0.5: недостижимо (база слишком мала)

Примечание: база событий в 24ч мала (1.5%/1.0%/0.3%), поэтому K с prec>=0.7
будут маленькими; для диспетчера главное — топ-K с известной ожидаемой долей.


### Как пользоваться
- Таблицы «top-K» дают прямой ответ: *открыть топ-N каналов → поймать M событий
  за 24ч, из них точных R*. Диспетчер задаёт K под свою пропускную способность.
- Калибровка показывает, в каких бинах `p24` доверять, а где вероятность
  завышена/занижена — туда дальше точечно направим isotonic per-bin.
